In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib qt

In [2]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from mackey_glass import get_mg_train_test_splits
from utils import validate_data_split_params
from optical_setup import OpticalSetup
from reservoir import Reservoir

In [3]:
mpl.rcParams.update({
    # Requires a LaTeX install (TeX Live / MiKTeX). Without one, set
    # 'text.usetex': False and 'mathtext.fontset': 'cm' instead.
    "pgf.texsystem"       : "xelatex",
    'text.usetex'         : False,
    'text.latex.preamble' : r'\usepackage{amsmath}',
    'font.family'         : 'serif',   # Computer Modern = default LaTeX font

    # Match your document's font sizes (most journals: 10 pt)
    'font.size'             : 10,
    'axes.labelsize'        : 10,
    'legend.title_fontsize' : 9.2,
    'xtick.labelsize'       : 9,
    'ytick.labelsize'       : 9,
    'legend.fontsize'       : 9,

    # Okabe–Ito palette — colorblind-safe, one line to replace the default cycle
    'axes.prop_cycle': mpl.cycler('color', [
        '#0072B2', '#56B4E9', '#D55E00', '#E69F00', 
        '#009E73', '#F0E442', '#CC79A7', '#000000'
    ]),

    'lines.linewidth'  : 1.5,
    'axes.linewidth'   : 0.8,
})

okabe_ito = [
        '#0072B2', '#56B4E9', '#D55E00', '#E69F00', 
        '#009E73', '#F0E442', '#CC79A7', '#000000'
    ]

mm = 1 / 25.4

# Mackey-Glass time series
---

In [5]:
data_split_params = {
    'train_phase': {
        'x0': 1.0,
        'seed': 42,
        'forget': 10, # Get rid of transient states
        'number_train_timesteps': 100, # Must be > forget
        'gap': 0,
    },
    'test_phase': {
        'prediction_horizon': 50, # Must be >= 1
        'number_forecast_origins': 10, # Must be >= 1
        'forecast_origin_spacing': 0, # Always >= 0. It can be different than zero if, and only if number_forecast_origins > 1
        'number_warmup_timesteps': 5 # Must be >= 0. Get rid of transient states
    }
}

forecasting_method = ['multi_step']
res_dim = 512
input_dim = 1

validate_data_split_params(params=data_split_params)

mg, X_train, y_train, X_test_warmup, X_test, y_test = get_mg_train_test_splits(data_split_params, return_complete_mg=True)

In [6]:
fig, axs = plt.subplots(
        figsize=(300 * mm, 100 * mm),
        layout='constrained',
    )

axs.set_title(rf"Data split (Initial conditions: {data_split_params['train_phase']['x0']}, seed: {data_split_params['train_phase']['seed']})")

# Train
# axs.plot(mg[0,:], mg[1,:], color='gray', alpha=0.2, marker='.')
axs.plot(X_train[0,:]*0.006, X_train[1,:], marker='.', label='X_train')

for i in range(data_split_params['test_phase']['prediction_horizon']):
    axs.plot(X_train[0,:data_split_params['train_phase']['number_train_timesteps']]*0.006, y_train[:, i], marker='.', label=rf'$y_{{target}}^{{h={i+1}}}$', linestyle=' ')
    if i == 1:
        break

# Test
axs.plot(X_test_warmup[0,:]*0.006, X_test_warmup[1,:], marker='.', label='X_test_warmup')
axs.plot(X_test[0,:]*0.006, X_test[1,:], marker='.', label='X_test')

N_o = data_split_params['test_phase']['number_forecast_origins']
s = data_split_params['test_phase']['forecast_origin_spacing'] # s=0 means no spacing (+1 in python)
rolling_window = N_o*(1+s)-s

for i in range(data_split_params['test_phase']['prediction_horizon']):
    axs.plot(X_test[0,:rolling_window]*0.006, y_test[:, i], marker='.', label=rf'$y_{{gt}}^{{h={i+1}}}$', linestyle=' ')
    if i == 1:
        break

axs.legend()
axs.set_xlabel(r"Lyapunov time ($\Lambda_{\max} t$)")
axs.set_ylabel(r'Mackey-Glass')
axs.spines[['top', 'right']].set_visible(False)
axs.minorticks_on()
axs.grid(which='major', linestyle='-', alpha=0.2)
axs.grid(which='minor', linestyle='--', alpha=0.1)
axs.set_ylim([0.0, 1.0])
axs.legend()
plt.show()

# Optical setup
---

In [ ]:
optical_setup = OpticalSetup(monitoring=True, grid_points=1024, state_nbin=16, res_dim=res_dim)

In [ ]:
optical_setup._on()

In [ ]:
optical_setup._dmd_warmup(horizon=10, criterion=0.80, period=2)
optical_setup._refresh_ref_speckle() #* Optional

In [ ]:
optical_setup._refresh_ref_speckle() #* Optional
optical_setup._check_stability(optical_path='linear')

In [ ]:
optical_setup._off()

# RC loop
---

In [ ]:
results = {}
optical_features = 'linear'

In [ ]:
optical_setup.reset_speckle_mem()

reservoir = Reservoir(res_dim=res_dim, input_dim=input_dim,
                      data_split_params=data_split_params, forecasting_method=forecasting_method,
                      leaky_rate=0.15, activation_func='norm255', encoding_func='identity',
                      reg_model='ridgeCV', reg_model_params={'fit_intercept': True, 'alphas': np.logspace(-8, 0), 'alpha_per_target': True, 'store_cv_results': False},
                      reg_window=None,
                      res_type='optical', optical_features=optical_features,
                      optical_setup=optical_setup,
                      seed=42)

reservoir.fit(train_data=X_train[1:,:], targets=y_train)
reservoir.predict(warmup_data=X_test_warmup[1:,:], test_data=X_test[1:,:], random_init=False)

results[optical_features] = {'y_test': y_test,
                             'reservoir_predictions': reservoir.predictions_multi}

if optical_features == 'linear':
    corr = np.array(optical_setup.corr_lin)
elif optical_features == 'nonlinear':
    corr = np.array(optical_setup.corr_nonlin)
else:
    raise ValueError('What kind of magical features are you looking for?')

if (corr > 0.985).all():
    print(f"Correlation min: {corr.min()}. Nice :)")
else:
    print(f"Correlation dropped to {corr.min()}. Sad ;-;")

In [ ]:
optical_features = 'linear'
sample_idx = 0

ground_truth = results[optical_features]['y_test'][sample_idx, :].squeeze()
prediction = results[optical_features]['reservoir_predictions'][sample_idx, :].squeeze()

timevec = np.arange(0, len(ground_truth)) * 0.006 # Maximal Lyapunov exponent

plt.plot(timevec, ground_truth, label='Ground Truth', color=okabe_ito[-1])
plt.plot(timevec, prediction, label='Ground Truth', color=okabe_ito[0])

plt.tight_layout()
plt.show()